In [4]:
import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Allow running from either the project root or the test folder.
ROOT = Path.cwd()
if not (ROOT / "src/allocations.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Pick up edits to allocations.py when this cell is rerun.
import src.allocations

importlib.reload(src.allocations)
from src.allocations import Allocations

In [ ]:
method = "test_bootstrap_nn"
# Variables
TEST_SIZE = 50
# 0.1, 0.25, 0.5
SELECTION_RATE = 0.5
# qualified_gagne_1, qualified_gagne_2, qualified_gagne_3
QUALIFICATION_COLUMN = "qualified_gagne_1"

NEED_COLUMN = "gagne_sum_t"
GROUP_COLUMN = "race"
RASHOMON_EPSILON = 0.01
ALLOCATION_DELTA = 0.1
NUM_MODELS = 100
ITERATIONS_PER_SPLIT = 25

fp_input = ROOT / "data/obermeyer/obermeyer_data_cleaned.csv"
fp_predictions = ROOT / f"data/predictions/{method}_{QUALIFICATION_COLUMN}_with_expected_needs.csv"
fp_output_opt = ROOT / f"data/results/allocations/opt_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_lda = ROOT / f"data/results/allocations/lda_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output_people = ROOT / f"data/results/allocations/people_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"
fp_output = ROOT / f"data/results/comparison_exp_need_{method}_y-{QUALIFICATION_COLUMN}_rate-{SELECTION_RATE}_size-{TEST_SIZE}.csv"

In [6]:
data = pd.read_csv(fp_input)
df = pd.read_csv(fp_predictions)
df = df.merge(
    data[["person_id", GROUP_COLUMN]],
    on="person_id",
    how="left",
    validate="many_to_one",
)

In [ ]:
a = Allocations(
    df=df,
    TEST_SIZE=TEST_SIZE,
    SELECTION_RATE=SELECTION_RATE,
    QUALIFICATION_COLUMN="y",
    NEED_COLUMN=NEED_COLUMN,
    GROUP_COLUMN=GROUP_COLUMN,
    RASHOMON_EPSILON=RASHOMON_EPSILON,
    ALLOCATION_EPSILON=ALLOCATION_DELTA,
    NUM_MODELS=NUM_MODELS,
    ITERATIONS_PER_SPLIT=ITERATIONS_PER_SPLIT,
)
allocations_opt, allocations_lda, people = a.get_allocations(expected_need_column="expected_need")
print(f"{len(people)} cohorts, {TEST_SIZE} people per cohort, k = {a.k}")

fp_output_opt.parent.mkdir(parents=True, exist_ok=True)
fp_output_lda.parent.mkdir(parents=True, exist_ok=True)
fp_output_people.parent.mkdir(parents=True, exist_ok=True)

allocations_opt.to_csv(fp_output_opt)
allocations_lda.to_csv(fp_output_lda)
people.to_csv(fp_output_people)

In [ ]:
all_results = []

for _, cohort_info in people.iterrows():
    seed = cohort_info["seed"]
    iteration = cohort_info["iteration"]

    split_df = df.loc[(df["seed"] == seed) & (df["person_id"] >= 0)]
    split_df = split_df.set_index("person_id")
    cohort = split_df.loc[cohort_info["people"]]

    for algorithm, allocations in [("OPT", allocations_opt), ("LDA", allocations_lda)]:
        cohort_allocations = allocations.loc[(allocations["seed"] == seed) & (allocations["iteration"] == iteration)]

        # Count selected sets, not model rows or allocation_idx values.
        # Sorting makes [person_a, person_b] and [person_b, person_a] identical.
        unique_allocations = {
            tuple(sorted(selected_ids)) for selected_ids in cohort_allocations["selected"]
        }
        model_names = {
            model for models in cohort_allocations["models"] for model in models
        }
        num_unique_allocations = len(unique_allocations)
        num_models = len(model_names)
        unique_allocation_ratio = num_unique_allocations / num_models if num_models else np.nan

        for _, allocation in cohort_allocations.iterrows():
            selected = cohort.loc[allocation["selected"]]

            mean_0 = selected.loc[selected[GROUP_COLUMN] == 0, NEED_COLUMN].mean()
            mean_1 = selected.loc[selected[GROUP_COLUMN] == 1, NEED_COLUMN].mean()
            absolute_disparity = abs(mean_1 - mean_0)
            disparity_ratio = np.nan
            if pd.notna(mean_0) and pd.notna(mean_1) and mean_0 != 0 and mean_1 != 0:
                disparity_ratio = max(mean_1 / mean_0, mean_0 / mean_1)

            for model in allocation["models"]:
                all_results.append({
                    "method": method,
                    "algorithm": algorithm,
                    "qualification_column": QUALIFICATION_COLUMN,
                    "seed": seed,
                    "iteration": iteration,
                    "model": model,
                    "k": a.k,
                    "epsilon": ALLOCATION_DELTA,
                    "tolerance_type": "add",
                    "expected_utility": selected[model].mean(),
                    "realized_utility": selected["y"].mean(),
                    "absolute_disparity": absolute_disparity,
                    "disparity_ratio": disparity_ratio,
                    "num_models": num_models,
                    "num_unique_allocations": num_unique_allocations,
                    "unique_allocation_ratio": unique_allocation_ratio,
                })

results = pd.DataFrame(all_results)
results

In [ ]:
fp_output.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(fp_output, index=False)
print(f"Saved {len(results)} rows to {fp_output}")